Image patch
   ↓
Shared CNN backbone (pretrained, mostly frozen)
   ↓
────────────────────────────────
│ Head 0: UB vs Lath-type      │  ← learned from pixels
│ Head 1: Deformed vs Non-def │  ← learned ONLY for UB
────────────────────────────────
   ↓
Ontology / rules (post-hoc)


In [1]:
import torch
import torch.nn as nn
import torchvision.models as models


In [2]:
class HierarchicalMicrostructureNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        # --- Backbone ---
        backbone = models.resnet18(pretrained=pretrained)
        self.feature_extractor = nn.Sequential(
            *list(backbone.children())[:-1]  # remove FC
        )
        self.feature_dim = backbone.fc.in_features

        # Freeze most layers
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # --- Head 0: Transformation family ---
        # UB vs Lath-type (M ± LB)
        self.head_family = nn.Linear(self.feature_dim, 2)

        # --- Head 1: Deformation state (UB only) ---
        self.head_deformation = nn.Linear(self.feature_dim, 2)

    def forward(self, x):
        feats = self.feature_extractor(x)
        feats = feats.view(feats.size(0), -1)

        out_family = self.head_family(feats)
        out_deformation = self.head_deformation(feats)

        return {
            "family": out_family,
            "deformation": out_deformation
        }

def hierarchical_loss(outputs, targets, criterion):
    """
    targets:
      family: 0 = UB, 1 = lath-type
      deformation: 0 = non-def, 1 = def (valid only if family == 0)
    """

    loss_family = criterion(outputs["family"], targets["family"])

    # Mask: only UB samples contribute to deformation loss
    ub_mask = (targets["family"] == 0)

    if ub_mask.any():
        loss_def = criterion(
            outputs["deformation"][ub_mask],
            targets["deformation"][ub_mask]
        )
    else:
        loss_def = torch.tensor(0.0, device=loss_family.device)

    return loss_family + loss_def


In [3]:
LABEL_MAP = {
    "dq_hollow": {
        "family": 1,        # lath-type (M ± LB)
        "deformation": -1   # not applicable
    },
    "iso": {
        "family": 0,        # upper bainite
        "deformation": 0    # non-deformed
    },
    "iso_deform": {
        "family": 0,        # upper bainite
        "deformation": 1    # deformed
    }
}


In [4]:
import numpy as np
import cv2

def extract_patches(img, patch_size=128, stride=64):
    patches = []
    h, w = img.shape[:2]

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = img[y:y+patch_size, x:x+patch_size]
            patches.append(patch)

    return patches


In [5]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from pathlib import Path


In [6]:
class MicrostructurePatchDataset(Dataset):
    def __init__(self, samples, patch_size=128, stride=64, augment=False):
        """
        samples: list of (image_path, label_dict)
        """
        self.items = []

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip() if augment else transforms.Lambda(lambda x: x),
            transforms.RandomVerticalFlip() if augment else transforms.Lambda(lambda x: x),
            transforms.ToTensor()
        ])

        for img_path, label in samples:
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

            patches = extract_patches(img, patch_size, stride)
            for p in patches:
                self.items.append((p, label))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        patch, label = self.items[idx]
        patch = self.transform(patch)

        family = torch.tensor(label["family"], dtype=torch.long)

        # deformation: dummy value if not applicable
        deformation = label["deformation"]
        if deformation == -1:
            deformation = 0

        deformation = torch.tensor(deformation, dtype=torch.long)

        return {
            "image": patch,
            "family": family,
            "deformation": deformation
        }


In [7]:
def collect_samples(data_root):
    samples = []

    for cls in LABEL_MAP.keys():
        cls_dir = Path(data_root) / cls
        for img_path in cls_dir.glob("*.*"):
            samples.append((img_path, LABEL_MAP[cls]))

    return samples


In [8]:
from sklearn.model_selection import KFold

def make_cv_loaders(
    data_root,
    batch_size=32,
    n_splits=5,
    patch_size=128,
    stride=64
):
    samples = collect_samples(data_root)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(samples)):
        train_samples = [samples[i] for i in train_idx]
        val_samples   = [samples[i] for i in val_idx]

        train_ds = MicrostructurePatchDataset(
            train_samples,
            patch_size=patch_size,
            stride=stride,
            augment=True
        )

        val_ds = MicrostructurePatchDataset(
            val_samples,
            patch_size=patch_size,
            stride=stride,
            augment=False
        )

        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )

        val_loader = torch.utils.data.DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        yield fold, train_loader, val_loader


In [9]:
for fold, train_loader, val_loader in make_cv_loaders("data_labeling/src/raw_images/attention"):
    print(f"Fold {fold}: {len(train_loader.dataset)} train patches")


Fold 0: 5712 train patches
Fold 1: 5712 train patches
Fold 2: 5712 train patches
Fold 3: 6120 train patches
Fold 4: 6120 train patches


In [ ]:
from tqdm import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = HierarchicalMicrostructureNet(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

num_epochs=20

for epoch in tqdm(range(num_epochs)):
    model.train()
    for batch in train_loader:
        images = batch["image"].to(device)
        targets = {
            "family": batch["family"].to(device),
            "deformation": batch["deformation"].to(device)
        }

        outputs = model(images)
        loss = hierarchical_loss(outputs, targets, criterion)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


c:\Users\hallo\Desktop\Bainitu_segmenation\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\hallo\Desktop\Bainitu_segmenation\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
  0%|          | 0/20 [00:00<?, ?it/s]c:\Users\hallo\Desktop\Bainitu_segmenation\venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
ub_mask = (targets["family"] == 0)


In [ ]:
def infer_final_label(pred):
    family = pred["family"]      # 0=UB, 1=lath-type
    deformation = pred["deformation"]

    if family == 1:
        return "Martensite ± Lower Bainite"

    if family == 0 and deformation == 1:
        return "Upper Bainite (deformed)"

    if family == 0 and deformation == 0:
        return "Upper Bainite (non-deformed)"
